# Real-world navigator: feature extraction + MLP training/evaluation

This notebook is structured to reproduce the navigator with the **real dataset**.

Assumptions fixed from the current specification:

- Only these visual paths are used:
  - `Almacen-Laboratorio_J`
  - `Cuarto_O-Laboratorio_M`
  - `Cubiculo_13-Laboratorio_I`
  - `Cubiculo_3-Laboratorio_E`
  - `Sala_E-Laboratorio_A`
- `registro_robot.csv` is ignored.
- The current trajectory comes from the `Simulation` folder.
- The keyframe comes from `visual_memory-1stdev`.
- `Log_Robot.csv` provides the label and the key RGB filename.
- `Idle` samples are discarded.
- Rotation is represented with **quaternions**.
- Temporal features are not used.
- Train/validation/test are split at the **visual-path level**, not row-wise.
- The notebook is designed to reuse your `modules/` pipeline once you paste the missing modules and LightGlue weights.

Run the notebook top-to-bottom. The first execution goal is to validate assumptions and expose the first bugs clearly.


In [1]:
from pathlib import Path
import os
import sys
import math
import time
import warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from PIL import Image

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    balanced_accuracy_score,
    f1_score,
)
from sklearn.dummy import DummyClassifier
from joblib import dump

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


## 1. Project configuration

Adjust these paths before the first run if needed.  
By default, the notebook assumes it is placed or executed from the project root.


In [2]:
# --- Project root ---
PROJECT_ROOT = Path.cwd()

# If needed, uncomment and edit:
# PROJECT_ROOT = Path("/home/rodriguez/Documents/GitHub/rpi_nav")

VISUAL_PATHS_ROOT = PROJECT_ROOT / "visual_paths"
MODULES_ROOT = PROJECT_ROOT / "modules"
OUTPUT_ROOT = PROJECT_ROOT / "notebook_outputs" / "real_nav_lightglue"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Only these folders are considered part of the dataset.
DATASET_PATHS = [
    "Almacen-Laboratorio_J",
    "Cuarto_O-Laboratorio_M",
    "Cubiculo_13-Laboratorio_I", # this path is broken
    "Cubiculo_3-Laboratorio_E",
    "Sala_E-Laboratorio_A",
]

# Path-level split.
TEST_PATHS = [
    "Cubiculo_13-Laboratorio_I", # this path is broken
    #"Cubiculo_3-Laboratorio_E",
]

# Optional validation path(s). If empty, validation will be created from the training rows.
VAL_PATHS = [
    "Cuarto_O-Laboratorio_M",
]

SIMULATION_FOLDER_NAME = "Simulation"
VISUAL_MEMORY_FOLDER_NAME = "visual_memory-1stdev"
LOG_FILENAME = "Log_Robot.csv"

DROP_LABELS = {"idle"}
LABEL_COLUMN = "action"

# RealSense D415 intrinsics
FX = 610.1170
FY = 610.2250
CX = 323.7142
CY = 237.8927
DEPTH_SCALE = 1.0           # set to 1e-3 if depth PNG values are in millimeters
INVALID_DEPTH_VALUE = 0.0

# Feature extraction behavior.
import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

print("torch:", torch.__version__)
print("torch.version.cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("device selected:", DEVICE)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
                 # use "cpu" first; revisit CUDA after the notebook runs end-to-end
LIGHTGLUE_MODE = "mnn"
MAX_SAMPLES_PER_PATH = None
SAVE_FEATURES_EVERY = 50

# Training.
FEATURE_COLUMNS = [
    "sim",
    "rmse",
    "x", "y", "z",
    "qw", "qx", "qy", "qz",
]

# Geometric features are extracted from T_source_to_target, where:
#   source = current simulation frame
#   target = key visual-memory frame
# Therefore x,y,z and qw,qx,qy,qz describe the motion current/source -> key/target.
RANDOM_STATE = 42

print("PROJECT_ROOT =", PROJECT_ROOT)
print("OUTPUT_ROOT  =", OUTPUT_ROOT)


torch: 2.5.1
torch.version.cuda: 12.1
cuda available: True
device selected: cuda
gpu: NVIDIA GeForce RTX 4050 Laptop GPU
PROJECT_ROOT = /home/rodriguez/Documents/GitHub/rpi_nav
OUTPUT_ROOT  = /home/rodriguez/Documents/GitHub/rpi_nav/notebook_outputs/real_nav_lightglue


## 2. Dataset audit

This checks that each selected visual path contains the expected files and folders.


In [3]:
def audit_one_path(base_dir: Path) -> Dict[str, object]:
    sim_dir = base_dir / SIMULATION_FOLDER_NAME
    vm_dir = base_dir / VISUAL_MEMORY_FOLDER_NAME
    log_csv = base_dir / LOG_FILENAME

    row = {
        "path_name": base_dir.name,
        "exists": base_dir.exists(),
        "simulation_dir": sim_dir.exists(),
        "simulation_rgb": (sim_dir / "rgb").exists(),
        "simulation_depth": (sim_dir / "depth").exists(),
        "visual_memory_dir": vm_dir.exists(),
        "visual_memory_rgb": (vm_dir / "rgb").exists(),
        "visual_memory_depth": (vm_dir / "depth").exists(),
        "log_csv": log_csv.exists(),
        "n_sim_rgb": None,
        "n_sim_depth": None,
        "n_vm_rgb": None,
        "n_vm_depth": None,
        "n_log_rows": None,
        "status": "OK",
        "note": "",
    }

    try:
        if (sim_dir / "rgb").exists():
            row["n_sim_rgb"] = len(list((sim_dir / "rgb").glob("*.png")))
        if (sim_dir / "depth").exists():
            row["n_sim_depth"] = len(list((sim_dir / "depth").glob("*.png")))
        if (vm_dir / "rgb").exists():
            row["n_vm_rgb"] = len(list((vm_dir / "rgb").glob("*.png")))
        if (vm_dir / "depth").exists():
            row["n_vm_depth"] = len(list((vm_dir / "depth").glob("*.png")))
        if log_csv.exists():
            row["n_log_rows"] = len(pd.read_csv(log_csv))
    except Exception as e:
        row["status"] = "ERROR"
        row["note"] = str(e)

    required = [
        row["simulation_dir"],
        row["simulation_rgb"],
        row["simulation_depth"],
        row["visual_memory_dir"],
        row["visual_memory_rgb"],
        row["visual_memory_depth"],
        row["log_csv"],
    ]
    if not all(required) and row["status"] == "OK":
        row["status"] = "MISSING"

    return row

audit_df = pd.DataFrame(
    [audit_one_path(VISUAL_PATHS_ROOT / name) for name in DATASET_PATHS]
)
display(audit_df)

assert len(audit_df) == len(DATASET_PATHS), "Audit table size mismatch."
assert (audit_df["status"] == "OK").all(), "At least one dataset path failed the audit."


,path_name,exists,simulation_dir,simulation_rgb,simulation_depth,visual_memory_dir,visual_memory_rgb,visual_memory_depth,log_csv,n_sim_rgb,n_sim_depth,n_vm_rgb,n_vm_depth,n_log_rows,status,note
0,Almacen-Laboratorio_J,True,True,True,True,True,True,True,True,2062,2062,72,72,2062,OK,
1,Cuarto_O-Laboratorio_M,True,True,True,True,True,True,True,True,930,930,30,30,930,OK,
2,Cubiculo_13-Laboratorio_I,True,True,True,True,True,True,True,True,1093,1093,32,32,1035,OK,
3,Cubiculo_3-Laboratorio_E,True,True,True,True,True,True,True,True,1524,1522,31,31,1522,OK,
4,Sala_E-Laboratorio_A,True,True,True,True,True,True,True,True,942,942,51,51,942,OK,


## 3. Module imports

This cell is meant to reuse your existing modules.  
If it fails, paste the modules in the expected structure and rerun only this cell.


In [4]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

rgbd_similarity_import_error = None
registration_import_error = None

try:
    from modules.rgbd_similarity import RGBDSimilarity
except Exception as e:
    RGBDSimilarity = None
    rgbd_similarity_import_error = e

try:
    from modules.feature_based_point_cloud_registration import FeatureBasedPointCloudRegistration
except Exception as e:
    FeatureBasedPointCloudRegistration = None
    registration_import_error = e

print("RGBDSimilarity import:", "OK" if RGBDSimilarity is not None else f"FAILED -> {rgbd_similarity_import_error}")
print("FeatureBasedPointCloudRegistration import:", "OK" if FeatureBasedPointCloudRegistration is not None else f"FAILED -> {registration_import_error}")

assert RGBDSimilarity is not None, "modules.rgbd_similarity is missing or incompatible."
assert FeatureBasedPointCloudRegistration is not None, "modules.feature_based_point_cloud_registration is missing or incompatible."


RGBDSimilarity import: OK
FeatureBasedPointCloudRegistration import: OK


/home/rodriguez/miniconda3/envs/habitat/lib/python3.9/site-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)
/home/rodriguez/Documents/GitHub/rpi_nav/lightglue/lightglue.py:24: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


## 4. Helper functions

These are generic notebook helpers.  
The actual matching and RGB-D similarity still come from your modules.


In [5]:
# ============================================================
# 4. Imports + helper utilities (corrected transform convention)
# ============================================================

from pathlib import Path
from typing import Dict, Any, Tuple

import numpy as np
import pandas as pd
import torch
import quaternion
from PIL import Image


def normalize_label(x: object) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip().lower()


def image_exists(path_like) -> bool:
    p = Path(path_like)
    return p.exists() and p.is_file()


def key_rgb_to_key_depth_filename(key_rgb_name: str) -> str:
    key_rgb_name = str(key_rgb_name).strip()
    if key_rgb_name.startswith("rgb_"):
        return key_rgb_name.replace("rgb_", "depth_", 1)
    return key_rgb_name.replace("rgb", "depth", 1)


def load_rgb(path) -> np.ndarray:
    return np.array(Image.open(path).convert("RGB"))


def load_depth(path) -> np.ndarray:
    return np.array(Image.open(path))


class RealFeatureBasedPointCloudRegistration(FeatureBasedPointCloudRegistration):
    def __init__(
        self,
        config: dict,
        device: str,
        id_run: int,
        feature_nav_conf: str = "LightGlue",
        feature_mode: str = "mnn",
        topological_map: bool = False,
        manual_operation: bool = False,
        fx: float = FX,
        fy: float = FY,
        cx: float = CX,
        cy: float = CY,
        depth_scale: float = DEPTH_SCALE,
        invalid_depth_value: float = INVALID_DEPTH_VALUE,
    ):
        super().__init__(
            config=config,
            device=device,
            id_run=id_run,
            feature_nav_conf=feature_nav_conf,
            feature_mode=feature_mode,
            topological_map=topological_map,
            manual_operation=manual_operation,
        )
        self.fx = float(fx)
        self.fy = float(fy)
        self.cx = float(cx)
        self.cy = float(cy)
        self.depth_scale = float(depth_scale)
        self.invalid_depth_value = float(invalid_depth_value)

    def generate_pc_in_cam_ref_frame(self, depth_img: np.ndarray, T_cam_world=None) -> np.ndarray:
        if depth_img.ndim != 2:
            raise ValueError(f"depth_img must be HxW, got shape {depth_img.shape}")

        depth = depth_img.astype(np.float64) * self.depth_scale
        h, w = depth.shape

        u, v = np.meshgrid(
            np.arange(w, dtype=np.float64),
            np.arange(h, dtype=np.float64),
        )

        Z = depth
        X = (u - self.cx) * Z / self.fx
        Y = (v - self.cy) * Z / self.fy

        valid = np.isfinite(Z) & (Z > self.invalid_depth_value)
        X[~valid] = np.nan
        Y[~valid] = np.nan
        Z[~valid] = np.nan

        ones = np.ones_like(Z, dtype=np.float64)
        pc_cam_h = np.stack([X, Y, Z, ones], axis=0).reshape(4, -1)
        return pc_cam_h

    def get_ipc_from_pc(self, pc_cam: np.ndarray, kp_cam: np.ndarray, h: int, w: int) -> np.ndarray:
        if pc_cam.ndim != 2 or pc_cam.shape[0] != 4:
            raise ValueError(f"pc_cam must have shape (4, H*W), got {pc_cam.shape}")

        kp = np.asarray(kp_cam, dtype=np.float64)
        if kp.ndim != 2 or kp.shape[1] != 2:
            raise ValueError(f"kp_cam must have shape (N,2), got {kp.shape}")

        xs = np.clip(np.round(kp[:, 0]).astype(int), 0, w - 1)
        ys = np.clip(np.round(kp[:, 1]).astype(int), 0, h - 1)

        cam_key_id = ys * w + xs
        ipc_cam_h = pc_cam[:, cam_key_id]
        return ipc_cam_h[:3].T

    def _filter_valid_depth_correspondences(
        self,
        kp_source: np.ndarray,
        kp_target: np.ndarray,
        source_depth: np.ndarray,
        target_depth: np.ndarray,
    ):
        hs, ws = source_depth.shape[:2]
        ht, wt = target_depth.shape[:2]

        ks = np.asarray(kp_source, dtype=np.float64)
        kt = np.asarray(kp_target, dtype=np.float64)

        xs = np.clip(np.round(ks[:, 0]).astype(int), 0, ws - 1)
        ys = np.clip(np.round(ks[:, 1]).astype(int), 0, hs - 1)

        xt = np.clip(np.round(kt[:, 0]).astype(int), 0, wt - 1)
        yt = np.clip(np.round(kt[:, 1]).astype(int), 0, ht - 1)

        ds = source_depth[ys, xs]
        dt = target_depth[yt, xt]

        valid = np.isfinite(ds) & np.isfinite(dt) & (ds > self.invalid_depth_value) & (dt > self.invalid_depth_value)
        return ks[valid], kt[valid]

    def compute_relative_pose(self, source_color, source_depth, target_color, target_depth):
        """
        Estimate the rigid transform that maps source 3D keypoints to target 3D keypoints.

        In this notebook:
            source = current simulation frame
            target = key visual-memory frame

        Therefore, the returned translation/quaternion describe the motion:
            current/source -> key/target
        """
        kp_source, kp_target = self.feature_nav.compute_matches(source_color, target_color)

        kp_source = np.asarray(kp_source)
        kp_target = np.asarray(kp_target)

        if len(kp_source) < 4 or len(kp_target) < 4:
            return True, None, None, None, None

        kp_source, kp_target = self._filter_valid_depth_correspondences(
            kp_source, kp_target, source_depth, target_depth
        )

        if len(kp_source) < 4 or len(kp_target) < 4:
            return True, None, None, None, None

        hs, ws = source_depth.shape[:2]
        ht, wt = target_depth.shape[:2]

        source_pc_h = self.generate_pc_in_cam_ref_frame(source_depth)
        target_pc_h = self.generate_pc_in_cam_ref_frame(target_depth)

        ipc_source = self.get_ipc_from_pc(source_pc_h, kp_source, hs, ws)
        ipc_target = self.get_ipc_from_pc(target_pc_h, kp_target, ht, wt)

        valid3d = np.isfinite(ipc_source).all(axis=1) & np.isfinite(ipc_target).all(axis=1)
        ipc_source = ipc_source[valid3d]
        ipc_target = ipc_target[valid3d]

        if len(ipc_source) < 4 or len(ipc_target) < 4:
            return True, None, None, None, None

        # est_T_source_to_target maps source/current keypoints onto target/key keypoints.
        rmse, transformed_ipc_source, est_T_source_to_target = self.execute_SVD_registration(ipc_source, ipc_target)
        R = est_T_source_to_target[:3, :3]
        t = est_T_source_to_target[:3, 3]
        q = quaternion.from_rotation_matrix(R)

        bot_lost = False
        return bot_lost, q, rmse, t, est_T_source_to_target


def build_models(
    device: str = DEVICE,
    id_run: int = 0,
    feature_nav_conf: str = "LightGlue",
    feature_mode: str = LIGHTGLUE_MODE,
    rgbd_similarity_threshold: float = 0.95,
    fx: float = FX,
    fy: float = FY,
    cx: float = CX,
    cy: float = CY,
    depth_scale: float = DEPTH_SCALE,
    invalid_depth_value: float = INVALID_DEPTH_VALUE,
) -> Tuple[RGBDSimilarity, RealFeatureBasedPointCloudRegistration]:
    config = {}

    rgbd_similarity = RGBDSimilarity(
        device=device,
        threshold=rgbd_similarity_threshold,
    )

    feature_registration = RealFeatureBasedPointCloudRegistration(
        config=config,
        device=device,
        id_run=id_run,
        feature_nav_conf=feature_nav_conf,
        feature_mode=feature_mode,
        topological_map=False,
        manual_operation=False,
        fx=fx,
        fy=fy,
        cx=cx,
        cy=cy,
        depth_scale=depth_scale,
        invalid_depth_value=invalid_depth_value,
    )

    return rgbd_similarity, feature_registration


def extract_pair_feature_vector(
    source_color: np.ndarray,
    source_depth: np.ndarray,
    target_color: np.ndarray,
    target_depth: np.ndarray,
    rgbd_similarity: RGBDSimilarity,
    feature_registration: RealFeatureBasedPointCloudRegistration,
) -> Dict[str, Any]:

    sim_score = rgbd_similarity.compute_image_similarity(
        source_color, source_depth, target_color, target_depth
    )

    result = feature_registration.compute_relative_pose(
        source_color, source_depth, target_color, target_depth
    )

    if len(result) == 5:
        bot_lost, est_quaternion_source_to_target, rmse, est_t_source_to_target, est_T_source_to_target = result
    elif len(result) == 4:
        bot_lost, est_quaternion_source_to_target, rmse, est_t_source_to_target = result
        est_T_source_to_target = None
    else:
        raise ValueError(f"Unexpected compute_relative_pose output length: {len(result)}")

    row = {
        "bot_lost": bool(bot_lost),
        "sim": float(sim_score),
        "rmse": np.nan if rmse is None else float(rmse),
        "x": np.nan,
        "y": np.nan,
        "z": np.nan,
        "qw": np.nan,
        "qx": np.nan,
        "qy": np.nan,
        "qz": np.nan,
    }

    if est_t_source_to_target is not None:
        t = np.asarray(est_t_source_to_target, dtype=np.float64).reshape(-1)
        if t.shape[0] >= 3:
            row["x"] = float(t[0])
            row["y"] = float(t[1])
            row["z"] = float(t[2])

    if est_quaternion_source_to_target is not None:
        row["qw"] = float(est_quaternion_source_to_target.w)
        row["qx"] = float(est_quaternion_source_to_target.x)
        row["qy"] = float(est_quaternion_source_to_target.y)
        row["qz"] = float(est_quaternion_source_to_target.z)

    return row


## 5. Build the pair table from logs

Each row links:
- current RGB-D frame from `Simulation/`
- key RGB-D frame from `visual_memory-1stdev/`
- label from `Log_Robot.csv`


In [6]:
def build_pairs_for_path(path_name: str) -> pd.DataFrame:
    base_dir = VISUAL_PATHS_ROOT / path_name

    sim_dir = base_dir / SIMULATION_FOLDER_NAME
    sim_rgb_dir = sim_dir / "rgb"
    sim_depth_dir = sim_dir / "depth"

    vm_dir = base_dir / VISUAL_MEMORY_FOLDER_NAME
    vm_rgb_dir = vm_dir / "rgb"
    vm_depth_dir = vm_dir / "depth"

    log_csv = base_dir / LOG_FILENAME

    log_df = pd.read_csv(log_csv).copy()
    log_df["label_norm"] = log_df[LABEL_COLUMN].map(normalize_label)

    # Discard Idle as requested
    log_df = log_df[~log_df["label_norm"].isin(DROP_LABELS)].copy()

    rows = []
    for idx, row in log_df.iterrows():
        current_rgb_name = str(row["rgb_filename"]).strip()
        current_depth_name = str(row["depth_filename"]).strip()
        key_rgb_name = str(row["key_image"]).strip()
        key_depth_name = key_rgb_to_key_depth_filename(key_rgb_name)

        current_rgb = sim_rgb_dir / current_rgb_name
        current_depth = sim_depth_dir / current_depth_name
        key_rgb = vm_rgb_dir / key_rgb_name
        key_depth = vm_depth_dir / key_depth_name

        rows.append({
            "path_name": path_name,
            "log_row_idx": int(idx),
            "timestamp_ms": row.get("timestamp_ms", np.nan),
            "datetime_iso": row.get("datetime_iso", ""),
            "label": row[LABEL_COLUMN],
            "label_norm": row["label_norm"],
            "simulation_dir": str(sim_dir),
            "visual_memory_dir": str(vm_dir),
            "current_rgb_filename": current_rgb_name,
            "current_depth_filename": current_depth_name,
            "key_rgb_filename": key_rgb_name,
            "key_depth_filename": key_depth_name,
            "current_rgb_path": str(current_rgb),
            "current_depth_path": str(current_depth),
            "key_rgb_path": str(key_rgb),
            "key_depth_path": str(key_depth),
            "simulation_dir_exists": sim_dir.exists(),
            "visual_memory_dir_exists": vm_dir.exists(),
            "current_rgb_exists": image_exists(current_rgb),
            "current_depth_exists": image_exists(current_depth),
            "key_rgb_exists": image_exists(key_rgb),
            "key_depth_exists": image_exists(key_depth),
        })

    return pd.DataFrame(rows)


pairs_df = pd.concat(
    [build_pairs_for_path(name) for name in DATASET_PATHS],
    ignore_index=True
)

display(pairs_df.head())

print("Rows after dropping Idle:", len(pairs_df))
print("\nLabel counts:")
display(
    pairs_df["label_norm"]
    .value_counts(dropna=False)
    .rename_axis("label")
    .reset_index(name="count")
)

missing_mask = ~(
    pairs_df["simulation_dir_exists"] &
    pairs_df["visual_memory_dir_exists"] &
    pairs_df["current_rgb_exists"] &
    pairs_df["current_depth_exists"] &
    pairs_df["key_rgb_exists"] &
    pairs_df["key_depth_exists"]
)

print("\nRows with missing files:", int(missing_mask.sum()))
display(
    pairs_df.loc[missing_mask, [
        "path_name",
        "log_row_idx",
        "label_norm",
        "current_rgb_filename",
        "current_depth_filename",
        "key_rgb_filename",
        "key_depth_filename",
        "simulation_dir_exists",
        "visual_memory_dir_exists",
        "current_rgb_exists",
        "current_depth_exists",
        "key_rgb_exists",
        "key_depth_exists",
    ]].head(20)
)


,path_name,log_row_idx,timestamp_ms,datetime_iso,label,label_norm,simulation_dir,visual_memory_dir,current_rgb_filename,current_depth_filename,key_rgb_filename,key_depth_filename,current_rgb_path,current_depth_path,key_rgb_path,key_depth_path,simulation_dir_exists,visual_memory_dir_exists,current_rgb_exists,current_depth_exists,key_rgb_exists,key_depth_exists
0,Almacen-Laboratorio_J,13,1772083572367,2026-02-25T21:26:12.367120,Update Memory,update memory,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,rgb_0013.png,depth_0013.png,rgb_0010.png,depth_0010.png,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,True,True,True,True,True,True
1,Almacen-Laboratorio_J,30,1772083574443,2026-02-25T21:26:14.443176,Update Memory,update memory,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,rgb_0030.png,depth_0030.png,rgb_0013.png,depth_0013.png,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,True,True,True,True,True,True
2,Almacen-Laboratorio_J,38,1772083575377,2026-02-25T21:26:15.377475,Forward,forward,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,rgb_0038.png,depth_0038.png,rgb_0013.png,depth_0013.png,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,True,True,True,True,True,True
3,Almacen-Laboratorio_J,39,1772083575507,2026-02-25T21:26:15.507214,Forward,forward,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,rgb_0039.png,depth_0039.png,rgb_0013.png,depth_0013.png,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,True,True,True,True,True,True
4,Almacen-Laboratorio_J,40,1772083575643,2026-02-25T21:26:15.643184,Forward,forward,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,rgb_0040.png,depth_0040.png,rgb_0013.png,depth_0013.png,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,True,True,True,True,True,True


Rows after dropping Idle: 1642

Label counts:


,label,count
0,forward,945
1,left,278
2,right,227
3,update memory,192



Rows with missing files: 0


,path_name,log_row_idx,label_norm,current_rgb_filename,current_depth_filename,key_rgb_filename,key_depth_filename,simulation_dir_exists,visual_memory_dir_exists,current_rgb_exists,current_depth_exists,key_rgb_exists,key_depth_exists


In [7]:
assert len(pairs_df) > 0, "No usable rows remained after dropping Idle."

assert (~(
    pairs_df["current_rgb_exists"] &
    pairs_df["current_depth_exists"] &
    pairs_df["key_rgb_exists"] &
    pairs_df["key_depth_exists"]
)).sum() < len(pairs_df), "Every row has a missing file. Fix keyframe references before proceeding."

usable_pairs_df = pairs_df[
    pairs_df["current_rgb_exists"] &
    pairs_df["current_depth_exists"] &
    pairs_df["key_rgb_exists"] &
    pairs_df["key_depth_exists"]
].reset_index(drop=True)

print("Usable rows:", len(usable_pairs_df))
display(
    usable_pairs_df["label_norm"]
    .value_counts(dropna=False)
    .rename_axis("label")
    .reset_index(name="count")
)


Usable rows: 1642


,label,count
0,forward,945
1,left,278
2,right,227
3,update memory,192


## 6. Quick visual sanity check

This checks a few current/key pairs before feature extraction.


In [8]:
def show_pair(i: int):
    row = usable_pairs_df.iloc[i]
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(load_rgb(Path(row["current_rgb_path"])))
    axes[0].set_title(f'Current\n{row["path_name"]}\n{row["current_rgb_filename"]}')
    axes[0].axis("off")

    axes[1].imshow(load_rgb(Path(row["key_rgb_path"])))
    axes[1].set_title(f'Key\n{row["key_rgb_filename"]}')
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()

for idx in [0, min(10, len(usable_pairs_df)-1), min(100, len(usable_pairs_df)-1)]:
    show_pair(idx)


## 7. Initialize the reused modules

This uses the actual pipeline modules.  
If this fails, the notebook is already in the right place to debug import or weight issues.


In [9]:
sim_model, matcher = build_models(device=DEVICE, depth_scale=1e-3)
print(type(sim_model).__name__, type(matcher).__name__)


/home/rodriguez/Documents/GitHub/rpi_nav/modules/rgbd_similarity.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.model.load_state_dict(torch.load(model_weights_pat

RGBDSimilarity RealFeatureBasedPointCloudRegistration


## 8. Single-sample dry run

Before extracting the full dataset, test one pair end-to-end.


In [10]:
def extract_one_feature_row(row: pd.Series, sim_model, matcher) -> Dict[str, object]:
    out = row.to_dict()
    out["status"] = "ok"
    out["fail_reason"] = ""

    out["sim"] = np.nan
    out["bot_lost"] = np.nan
    out["rmse"] = np.nan
    out["x"] = np.nan
    out["y"] = np.nan
    out["z"] = np.nan
    out["qw"] = np.nan
    out["qx"] = np.nan
    out["qy"] = np.nan
    out["qz"] = np.nan

    try:
        s_rgb = load_rgb(Path(row["current_rgb_path"]))
        s_depth = load_depth(Path(row["current_depth_path"]))
        t_rgb = load_rgb(Path(row["key_rgb_path"]))
        t_depth = load_depth(Path(row["key_depth_path"]))
    except Exception as e:
        out["status"] = "failed"
        out["fail_reason"] = f"load_error: {e}"
        return out

    try:
        sim = sim_model.compute_image_similarity(s_rgb, s_depth, t_rgb, t_depth)
        out["sim"] = float(sim)
    except Exception as e:
        out["status"] = "failed"
        out["fail_reason"] = f"similarity_error: {e}"
        return out

    try:
        result = matcher.compute_relative_pose(s_rgb, s_depth, t_rgb, t_depth)

        if len(result) == 5:
            bot_lost, est_quaternion_source_to_target, rmse, est_t_source_to_target, est_T_source_to_target = result
        elif len(result) == 4:
            bot_lost, est_quaternion_source_to_target, rmse, est_t_source_to_target = result
            est_T_source_to_target = None
        else:
            out["status"] = "failed"
            out["fail_reason"] = f"unexpected_relative_pose_output_len: {len(result)}"
            return out

        out["bot_lost"] = bool(bot_lost)
        out["rmse"] = np.nan if rmse is None else float(rmse)

        if est_t_source_to_target is not None:
            t = np.asarray(est_t_source_to_target, dtype=np.float64).reshape(-1)
            if t.shape[0] >= 3:
                out["x"] = float(t[0])
                out["y"] = float(t[1])
                out["z"] = float(t[2])

        if est_quaternion_source_to_target is not None:
            out["qw"] = float(est_quaternion_source_to_target.w)
            out["qx"] = float(est_quaternion_source_to_target.x)
            out["qy"] = float(est_quaternion_source_to_target.y)
            out["qz"] = float(est_quaternion_source_to_target.z)

    except Exception as e:
        out["status"] = "failed"
        out["fail_reason"] = f"registration_error: {e}"
        return out

    return out

dry_idx = 0
dry_row = usable_pairs_df.iloc[dry_idx]
dry_features = extract_one_feature_row(dry_row, sim_model, matcher)
dry_features


{'path_name': 'Almacen-Laboratorio_J',
 'log_row_idx': 13,
 'timestamp_ms': 1772083572367,
 'datetime_iso': '2026-02-25T21:26:12.367120',
 'label': 'Update Memory',
 'label_norm': 'update memory',
 'simulation_dir': '/home/rodriguez/Documents/GitHub/rpi_nav/visual_paths/Almacen-Laboratorio_J/Simulation',
 'visual_memory_dir': '/home/rodriguez/Documents/GitHub/rpi_nav/visual_paths/Almacen-Laboratorio_J/visual_memory-1stdev',
 'current_rgb_filename': 'rgb_0013.png',
 'current_depth_filename': 'depth_0013.png',
 'key_rgb_filename': 'rgb_0010.png',
 'key_depth_filename': 'depth_0010.png',
 'current_rgb_path': '/home/rodriguez/Documents/GitHub/rpi_nav/visual_paths/Almacen-Laboratorio_J/Simulation/rgb/rgb_0013.png',
 'current_depth_path': '/home/rodriguez/Documents/GitHub/rpi_nav/visual_paths/Almacen-Laboratorio_J/Simulation/depth/depth_0013.png',
 'key_rgb_path': '/home/rodriguez/Documents/GitHub/rpi_nav/visual_paths/Almacen-Laboratorio_J/visual_memory-1stdev/rgb/rgb_0010.png',
 'key_depth_

## 9. Full feature extraction

This creates two outputs:

- `features_full.csv`: all attempted rows with failure reasons
- `features_trainable.csv`: only successful rows


In [11]:
feature_rows = []
t0 = time.time()

iter_df = usable_pairs_df.copy()
if MAX_SAMPLES_PER_PATH is not None:
    iter_df = (
        iter_df.groupby("path_name", group_keys=False)
        .head(MAX_SAMPLES_PER_PATH)
        .reset_index(drop=True)
    )

for i, (_, row) in enumerate(iter_df.iterrows(), start=1):
    feat = extract_one_feature_row(row, sim_model, matcher)
    feature_rows.append(feat)

    if (i % SAVE_FEATURES_EVERY == 0) or (i == len(iter_df)):
        tmp_df = pd.DataFrame(feature_rows)
        tmp_df.to_csv(OUTPUT_ROOT / "features_full_partial.csv", index=False)
        print(f"[{i:04d}/{len(iter_df):04d}] checkpoint saved")

features_full_df = pd.DataFrame(feature_rows)
features_full_path = OUTPUT_ROOT / "features_full.csv"
features_full_df.to_csv(features_full_path, index=False)

features_trainable_df = features_full_df[features_full_df["status"] == "ok"].copy()
features_trainable_path = OUTPUT_ROOT / "features_trainable.csv"
features_trainable_df.to_csv(features_trainable_path, index=False)

elapsed = time.time() - t0
print(f"Done in {elapsed/60:.2f} min")
print("Saved:", features_full_path)
print("Saved:", features_trainable_path)


[0050/1642] checkpoint saved
[0100/1642] checkpoint saved
[0150/1642] checkpoint saved
[0200/1642] checkpoint saved
[0250/1642] checkpoint saved
[0300/1642] checkpoint saved
[0350/1642] checkpoint saved
[0400/1642] checkpoint saved
[0450/1642] checkpoint saved
[0500/1642] checkpoint saved
[0550/1642] checkpoint saved
[0600/1642] checkpoint saved
[0650/1642] checkpoint saved
[0700/1642] checkpoint saved
[0750/1642] checkpoint saved
[0800/1642] checkpoint saved
[0850/1642] checkpoint saved
[0900/1642] checkpoint saved
[0950/1642] checkpoint saved
[1000/1642] checkpoint saved
[1050/1642] checkpoint saved
[1100/1642] checkpoint saved
[1150/1642] checkpoint saved
[1200/1642] checkpoint saved
[1250/1642] checkpoint saved
[1300/1642] checkpoint saved
[1350/1642] checkpoint saved
[1400/1642] checkpoint saved
[1450/1642] checkpoint saved
[1500/1642] checkpoint saved
[1550/1642] checkpoint saved
[1600/1642] checkpoint saved
[1642/1642] checkpoint saved
Done in 5.91 min
Saved: /home/rodriguez/Doc

In [12]:
print("Attempted rows:", len(features_full_df))
print("Successful rows:", len(features_trainable_df))
print("Success rate:", round(len(features_trainable_df) / max(len(features_full_df), 1), 4))

print("\nFailure reasons:")
display(
    features_full_df.loc[features_full_df["status"] != "ok", "fail_reason"]
    .value_counts(dropna=False)
    .rename_axis("fail_reason")
    .reset_index(name="count")
)

print("\nSuccessful labels:")
display(
    features_trainable_df["label_norm"]
    .value_counts(dropna=False)
    .rename_axis("label")
    .reset_index(name="count")
)

display(features_trainable_df.head())


Attempted rows: 1642
Successful rows: 1642
Success rate: 1.0

Failure reasons:


,fail_reason,count



Successful labels:


,label,count
0,forward,945
1,left,278
2,right,227
3,update memory,192


,path_name,log_row_idx,timestamp_ms,datetime_iso,label,label_norm,simulation_dir,visual_memory_dir,current_rgb_filename,current_depth_filename,key_rgb_filename,key_depth_filename,current_rgb_path,current_depth_path,key_rgb_path,key_depth_path,simulation_dir_exists,visual_memory_dir_exists,current_rgb_exists,current_depth_exists,key_rgb_exists,key_depth_exists,status,fail_reason,sim,bot_lost,rmse,x,y,z,qw,qx,qy,qz
0,Almacen-Laboratorio_J,13,1772083572367,2026-02-25T21:26:12.367120,Update Memory,update memory,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,rgb_0013.png,depth_0013.png,rgb_0010.png,depth_0010.png,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,True,True,True,True,True,True,ok,,0.769632,False,0.717583,0.123125,-0.195644,-0.894291,0.998259,-0.012937,0.056632,-0.010197
1,Almacen-Laboratorio_J,30,1772083574443,2026-02-25T21:26:14.443176,Update Memory,update memory,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,rgb_0030.png,depth_0030.png,rgb_0013.png,depth_0013.png,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,True,True,True,True,True,True,ok,,0.787968,False,1.219702,-0.068670,-0.666228,-1.772251,0.995320,-0.049125,0.056841,-0.060770
2,Almacen-Laboratorio_J,38,1772083575377,2026-02-25T21:26:15.377475,Forward,forward,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,rgb_0038.png,depth_0038.png,rgb_0013.png,depth_0013.png,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,True,True,True,True,True,True,ok,,0.773076,False,1.017799,0.350364,-0.180706,-1.474778,0.999766,-0.002818,0.009392,-0.019282
3,Almacen-Laboratorio_J,39,1772083575507,2026-02-25T21:26:15.507214,Forward,forward,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,rgb_0039.png,depth_0039.png,rgb_0013.png,depth_0013.png,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,True,True,True,True,True,True,ok,,0.772637,False,0.807778,0.155365,0.044004,-1.623744,0.999196,0.018878,0.034815,-0.006250
4,Almacen-Laboratorio_J,40,1772083575643,2026-02-25T21:26:15.643184,Forward,forward,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,rgb_0040.png,depth_0040.png,rgb_0013.png,depth_0013.png,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,True,True,True,True,True,True,ok,,0.781615,False,0.766848,0.142559,0.031706,-1.613428,0.999161,0.017964,0.036039,-0.007467


## 10. Split by visual path

- test paths are fixed by `TEST_PATHS`
- optional validation paths are fixed by `VAL_PATHS`
- all remaining paths are used for training


In [13]:
all_path_names = sorted(features_trainable_df["path_name"].unique().tolist())

train_paths = [p for p in all_path_names if p not in set(TEST_PATHS) | set(VAL_PATHS)]
val_paths = [p for p in VAL_PATHS if p in all_path_names]
test_paths = [p for p in TEST_PATHS if p in all_path_names]

print("Available:", all_path_names)
print("Train    :", train_paths)
print("Val      :", val_paths)
print("Test     :", test_paths)

assert len(test_paths) >= 1, "At least one test path must be present in the successful dataset."
assert len(train_paths) >= 1, "At least one training path must remain."

train_df = features_trainable_df[features_trainable_df["path_name"].isin(train_paths)].copy()

if len(val_paths) > 0:
    val_df = features_trainable_df[features_trainable_df["path_name"].isin(val_paths)].copy()
else:
    # Fallback only if no validation path was specified.
    train_df, val_df = train_test_split(
        train_df,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=train_df["label_norm"] if train_df["label_norm"].nunique() > 1 else None,
    )

test_df = features_trainable_df[features_trainable_df["path_name"].isin(test_paths)].copy()

for name, df_ in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"\n{name}: n={len(df_)}")
    display(df_["label_norm"].value_counts(dropna=False).rename_axis("label").reset_index(name="count"))


Available: ['Almacen-Laboratorio_J', 'Cuarto_O-Laboratorio_M', 'Cubiculo_13-Laboratorio_I', 'Cubiculo_3-Laboratorio_E', 'Sala_E-Laboratorio_A']
Train    : ['Almacen-Laboratorio_J', 'Cubiculo_3-Laboratorio_E', 'Sala_E-Laboratorio_A']
Val      : ['Cuarto_O-Laboratorio_M']
Test     : ['Cubiculo_13-Laboratorio_I']

train: n=1088


,label,count
0,forward,622
1,left,168
2,right,167
3,update memory,131



val: n=231


,label,count
0,forward,152
1,update memory,31
2,left,27
3,right,21



test: n=323


,label,count
0,forward,171
1,left,83
2,right,39
3,update memory,30


## 11. Train the navigator

Baseline:
- StandardScaler
- MLP classifier
- comparison against majority-class dummy classifier


In [14]:
train_df = train_df.dropna(subset=FEATURE_COLUMNS).copy()
val_df   = val_df.dropna(subset=FEATURE_COLUMNS).copy()
test_df  = test_df.dropna(subset=FEATURE_COLUMNS).copy()


In [15]:
X_train = train_df[FEATURE_COLUMNS].astype(float).copy()
X_val = val_df[FEATURE_COLUMNS].astype(float).copy()
X_test = test_df[FEATURE_COLUMNS].astype(float).copy()

'''
print("NaNs per split:")
print("train:\n", X_train.isna().sum())
print("val:\n", X_val.isna().sum())
print("test:\n", X_test.isna().sum())

print("\nRows with any NaN in train:", X_train.isna().any(axis=1).sum())
print("Rows with any NaN in val:", X_val.isna().any(axis=1).sum())
print("Rows with any NaN in test:", X_test.isna().any(axis=1).sum())
'''

y_train = train_df["label_norm"].copy()
y_val = val_df["label_norm"].copy()
y_test = test_df["label_norm"].copy()

label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_val_enc = label_encoder.transform(y_val)
y_test_enc = label_encoder.transform(y_test)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc = scaler.transform(X_val)
X_test_sc = scaler.transform(X_test)

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train_sc, y_train_enc)

mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    batch_size="auto",
    learning_rate="adaptive",
    max_iter=1200,
    random_state=RANDOM_STATE,
)

mlp.fit(X_train_sc, y_train_enc)

dump(scaler, OUTPUT_ROOT / "scaler.joblib")
dump(label_encoder, OUTPUT_ROOT / "label_encoder.joblib")
dump(mlp, OUTPUT_ROOT / "mlp_lightglue_rgbd_real.joblib")

print("Artifacts saved to", OUTPUT_ROOT)


Artifacts saved to /home/rodriguez/Documents/GitHub/rpi_nav/notebook_outputs/real_nav_lightglue


## 12. Evaluation helpers


In [16]:
def evaluate_classifier(name: str, clf, X, y_true_enc, class_names):
    y_pred_enc = clf.predict(X)
    y_true = class_names[y_true_enc]
    y_pred = class_names[y_pred_enc]

    macro_f1 = f1_score(y_true_enc, y_pred_enc, average="macro")
    weighted_f1 = f1_score(y_true_enc, y_pred_enc, average="weighted")
    bal_acc = balanced_accuracy_score(y_true_enc, y_pred_enc)

    print(f"===== {name} =====")
    print(f"Balanced accuracy: {bal_acc:.4f}")
    print(f"Macro F1         : {macro_f1:.4f}")
    print(f"Weighted F1      : {weighted_f1:.4f}")
    print()
    print(classification_report(y_true, y_pred, digits=4))

    cm = confusion_matrix(y_true, y_pred, labels=list(class_names))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(class_names))
    fig, ax = plt.subplots(figsize=(6, 6))
    disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
    ax.set_title(name)
    plt.show()

    return pd.DataFrame({"y_true": y_true, "y_pred": y_pred})

class_names = label_encoder.classes_
print("Classes:", list(class_names))


Classes: ['forward', 'left', 'right', 'update memory']


In [17]:
pred_val_dummy = evaluate_classifier("Dummy / val", dummy, X_val_sc, y_val_enc, class_names)
pred_val_mlp = evaluate_classifier("MLP / val", mlp, X_val_sc, y_val_enc, class_names)


===== Dummy / val =====
Balanced accuracy: 0.2500
Macro F1         : 0.1984
Weighted F1      : 0.5223

               precision    recall  f1-score   support

      forward     0.6580    1.0000    0.7937       152
         left     0.0000    0.0000    0.0000        27
        right     0.0000    0.0000    0.0000        21
update memory     0.0000    0.0000    0.0000        31

     accuracy                         0.6580       231
    macro avg     0.1645    0.2500    0.1984       231
 weighted avg     0.4330    0.6580    0.5223       231

===== MLP / val =====
Balanced accuracy: 0.4146
Macro F1         : 0.3576
Weighted F1      : 0.5152

               precision    recall  f1-score   support

      forward     0.7339    0.5987    0.6594       152
         left     0.1951    0.2963    0.2353        27
        right     0.3043    0.6667    0.4179        21
update memory     0.1500    0.0968    0.1176        31

     accuracy                         0.5022       231
    macro avg     0.3

In [18]:
pred_test_dummy = evaluate_classifier("Dummy / test", dummy, X_test_sc, y_test_enc, class_names)
pred_test_mlp = evaluate_classifier("MLP / test", mlp, X_test_sc, y_test_enc, class_names)


===== Dummy / test =====
Balanced accuracy: 0.2500
Macro F1         : 0.1731
Weighted F1      : 0.3665

               precision    recall  f1-score   support

      forward     0.5294    1.0000    0.6923       171
         left     0.0000    0.0000    0.0000        83
        right     0.0000    0.0000    0.0000        39
update memory     0.0000    0.0000    0.0000        30

     accuracy                         0.5294       323
    macro avg     0.1324    0.2500    0.1731       323
 weighted avg     0.2803    0.5294    0.3665       323

===== MLP / test =====
Balanced accuracy: 0.3742
Macro F1         : 0.3797
Weighted F1      : 0.4884

               precision    recall  f1-score   support

      forward     0.6292    0.6550    0.6418       171
         left     0.4583    0.2651    0.3359        83
        right     0.4571    0.4103    0.4324        39
update memory     0.0806    0.1667    0.1087        30

     accuracy                         0.4799       323
    macro avg     0

## 13. Attach predictions back to the samples


In [19]:
test_results_df = test_df.reset_index(drop=True).copy()
test_results_df["y_true"] = class_names[y_test_enc]
test_results_df["y_pred"] = class_names[mlp.predict(X_test_sc)]
test_results_df["is_correct"] = test_results_df["y_true"] == test_results_df["y_pred"]

test_results_path = OUTPUT_ROOT / "test_predictions.csv"
test_results_df.to_csv(test_results_path, index=False)

print("Saved:", test_results_path)
display(test_results_df.head())


Saved: /home/rodriguez/Documents/GitHub/rpi_nav/notebook_outputs/real_nav_lightglue/test_predictions.csv


,path_name,log_row_idx,timestamp_ms,datetime_iso,label,label_norm,simulation_dir,visual_memory_dir,current_rgb_filename,current_depth_filename,key_rgb_filename,key_depth_filename,current_rgb_path,current_depth_path,key_rgb_path,key_depth_path,simulation_dir_exists,visual_memory_dir_exists,current_rgb_exists,current_depth_exists,key_rgb_exists,key_depth_exists,status,fail_reason,sim,bot_lost,rmse,x,y,z,qw,qx,qy,qz,y_true,y_pred,is_correct
0,Cubiculo_13-Laboratorio_I,0,1772084209830,2026-02-25T21:36:49,Forward,forward,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,rgb_0058.png,depth_0058.png,rgb_0030.png,depth_0030.png,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,True,True,True,True,True,True,ok,,0.816397,False,1.077789,0.472060,0.090576,-1.832279,-0.997967,-0.017549,0.061268,0.001005,forward,update memory,False
1,Cubiculo_13-Laboratorio_I,1,1772084209936,2026-02-25T21:36:49,Forward,forward,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,rgb_0059.png,depth_0059.png,rgb_0030.png,depth_0030.png,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,True,True,True,True,True,True,ok,,0.816683,False,1.103495,0.474662,0.217898,-1.834997,-0.997840,-0.023139,0.061451,0.001991,forward,update memory,False
2,Cubiculo_13-Laboratorio_I,2,1772084210062,2026-02-25T21:36:50,Forward,forward,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,rgb_0060.png,depth_0060.png,rgb_0030.png,depth_0030.png,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,True,True,True,True,True,True,ok,,0.816118,False,1.072573,0.600387,0.227475,-1.733701,-0.997430,-0.024100,0.067328,0.004506,forward,update memory,False
3,Cubiculo_13-Laboratorio_I,3,1772084210202,2026-02-25T21:36:50,Forward,forward,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,rgb_0061.png,depth_0061.png,rgb_0030.png,depth_0030.png,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,True,True,True,True,True,True,ok,,0.811908,False,1.049617,0.780659,0.164133,-1.616810,-0.995542,-0.026014,0.090636,0.002310,forward,forward,True
4,Cubiculo_13-Laboratorio_I,4,1772084210330,2026-02-25T21:36:50,Forward,forward,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,rgb_0062.png,depth_0062.png,rgb_0030.png,depth_0030.png,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,/home/rodriguez/Documents/GitHub/rpi_nav/visua...,True,True,True,True,True,True,ok,,0.813417,False,1.385679,0.262653,-0.651995,-1.528668,-0.996850,0.031889,0.066837,-0.028381,forward,forward,True


## 14. Quick qualitative inspection of errors


In [20]:
wrong_df = test_results_df.loc[~test_results_df["is_correct"]].copy()
print("Wrong predictions:", len(wrong_df))

cols_to_show = [
    "path_name", "current_rgb_filename", "key_rgb_filename", "y_true", "y_pred",
    "sim", "rmse"
]
available_cols = [c for c in cols_to_show if c in wrong_df.columns]
display(wrong_df[available_cols].head(20))


Wrong predictions: 168


,path_name,current_rgb_filename,key_rgb_filename,y_true,y_pred,sim,rmse
0,Cubiculo_13-Laboratorio_I,rgb_0058.png,rgb_0030.png,forward,update memory,0.816397,1.077789
1,Cubiculo_13-Laboratorio_I,rgb_0059.png,rgb_0030.png,forward,update memory,0.816683,1.103495
2,Cubiculo_13-Laboratorio_I,rgb_0060.png,rgb_0030.png,forward,update memory,0.816118,1.072573
12,Cubiculo_13-Laboratorio_I,rgb_0089.png,rgb_0037.png,forward,update memory,0.807316,1.374928
13,Cubiculo_13-Laboratorio_I,rgb_0090.png,rgb_0037.png,forward,update memory,0.805063,1.265085
14,Cubiculo_13-Laboratorio_I,rgb_0091.png,rgb_0037.png,forward,update memory,0.805063,1.265085
18,Cubiculo_13-Laboratorio_I,rgb_0095.png,rgb_0037.png,forward,update memory,0.802845,1.532326
19,Cubiculo_13-Laboratorio_I,rgb_0096.png,rgb_0037.png,forward,update memory,0.802844,1.295391
20,Cubiculo_13-Laboratorio_I,rgb_0097.png,rgb_0037.png,forward,update memory,0.810509,1.212973
21,Cubiculo_13-Laboratorio_I,rgb_0098.png,rgb_0037.png,forward,update memory,0.810509,1.212973


In [21]:
def show_test_case(case_df: pd.DataFrame, idx: int = 0):
    row = case_df.iloc[idx]
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(load_rgb(Path(row["current_rgb_path"])))
    axes[0].set_title(
        f"Current\n{row['path_name']}\ntrue={row['y_true']} pred={row['y_pred']}"
    )
    axes[0].axis("off")

    axes[1].imshow(load_rgb(Path(row["key_rgb_path"])))
    axes[1].set_title(
        f"Key\nsim={row['sim']:.3f}\nrmse={row['rmse']:.3f}"
    )
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()

if len(wrong_df) > 0:
    show_test_case(wrong_df, 0)


## 15. Final notes

Expected first-run failure points:

1. `FeatureMatcher` import or LightGlue weights loading
2. mismatch between `FeatureMatcher.compute_matches(...)` expected inputs and the notebook call
3. mismatch between `RGBDSimilarity` interface and the notebook call
4. current-frame location inside `Simulation/` if the final structure differs
5. key depth filename rule if it is not `rgb_XXXX.png -> depth_XXXX.png`
6. camera intrinsics if the SVD geometry requires calibrated projection instead of the current approximation

The notebook is intentionally explicit so each failure appears in one localized cell instead of being hidden.
